# Train Fidgety trên Kaggle — đọc trực tiếp data.zip

1. Tạo Kaggle Dataset chứa **data.zip**, thêm Dataset vào notebook bằng **Add Input**.
2. Import notebook này; bật **Internet** và chọn **GPU** trong Settings.
3. Chạy các cell theo thứ tự. Mặc định: **VID, fold 1, 1 run, 5 epoch** để thử.
4. Khi chạy thử thành công, đổi `EXPERIMENT_NAME`, `FOLDS`, `RUNS`, `EPOCHS` trong cell cấu hình rồi chạy lại từ cell đó.

Không giải nén ZIP, không copy data sang `/kaggle/working`. Mỗi file TXT được đọc từ ZIP vào RAM. RAM vẫn phải chứa các mảng của một fold; xử lý dữ liệu giữ nguyên tác giả.

Checkpoint của phiên Kaggle này cần huấn luyện mới; `.hf5` của Keras không dùng được. Bản PyTorch không đảm bảo kết quả số giống hệt TensorFlow.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path("/kaggle/working/Fidgety")
if not SOURCE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(SOURCE_DIR / "requirements.txt")], check=True)
REVISION = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", REVISION)


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings trước khi train.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
# Code tự dùng một GPU; không tự phân phối sang GPU thứ hai.


## Cấu hình

`MODEL`: `vid`, `imu`, `mat`, hoặc `fusion`.

- Chạy thử: `FOLDS = [1]`, `RUNS = 1`, `EPOCHS = 5`.
- Chạy đầy đủ như script best-model: `FOLDS = list(range(1, 10))`, `RUNS = 20`, `EPOCHS = 1000`. Có early stopping patience 10.
- Chọn tên `EXPERIMENT_NAME` mới khi đổi cấu hình hoặc source.
- Nếu tìm thấy nhiều ZIP, điền chính xác `DATA_ZIP`.

Notebook dùng kiến trúc best-model có sẵn, không chạy lại dò siêu tham số.


In [ ]:
import json
import os
import zipfile

DATA_ZIP = ""  # Ví dụ: /kaggle/input/datasets/username/dataset/data.zip
MODEL = "vid"
FOLDS = [1]
RUNS = 1
EPOCHS = 5
EXPERIMENT_NAME = "vid_smoke_01"
SKIP_COMPLETED_FOLDS = True

SCRIPTS = {
    "vid": "vitpose_kpc_vel_conv_best_model_9fold.py",
    "imu": "imu_acc_gyr_conv_best_model_9fold.py",
    "mat": "mat_conv_best_model_9fold.py",
    "fusion": "mat_vitpose_imu_conv_best_model_9fold.py",
}
assert MODEL in SCRIPTS
assert FOLDS and len(set(FOLDS)) == len(FOLDS) and all(isinstance(f, int) and 1 <= f <= 9 for f in FOLDS)
assert RUNS > 0 and EPOCHS > 0
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME
if not DATA_ZIP:
    candidates = sorted(Path("/kaggle/input").rglob("*.zip"))
    print("Các ZIP tìm thấy:", *candidates, sep="\n")
    if len(candidates) != 1:
        raise ValueError("Điền DATA_ZIP để chọn đúng file ZIP.")
    DATA_ZIP = str(candidates[0])
DATA_ZIP = str(Path(DATA_ZIP).resolve(strict=True))
OUTPUT_DIR = Path("/kaggle/working/outputs") / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = dict(model=MODEL, folds=FOLDS, runs=RUNS, epochs=EPOCHS,
              data_zip=DATA_ZIP, data_zip_bytes=Path(DATA_ZIP).stat().st_size, source_commit=REVISION)
config_path = OUTPUT_DIR / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise ValueError("Thư mục này chứa cấu hình khác. Hãy đổi EXPERIMENT_NAME.")
config_path.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print("Output:", OUTPUT_DIR)


## Kiểm tra ZIP và dữ liệu đầu vào

Chỉ đọc danh mục ZIP và một file nhãn nhỏ, không giải nén dữ liệu.


In [ ]:
with zipfile.ZipFile(DATA_ZIP) as archive:
    infos = archive.infolist()
    print("ZIP GiB:", round(Path(DATA_ZIP).stat().st_size / 1024**3, 2))
    print("Kích thước nếu giải nén GiB:", round(sum(i.file_size for i in infos) / 1024**3, 2))
    print("Số mục:", len(infos))
    print("\n".join(i.filename for i in infos[:12]))

sys.path.insert(0, str(SOURCE_DIR))
from data_io import loadtxt
os.environ["FIDGETY_DATA_ZIP"] = DATA_ZIP

modalities = ["mat", "imu", "vitpose_kpc"] if MODEL == "fusion" else [{"vid": "vitpose_kpc", "imu": "imu", "mat": "mat"}[MODEL]]
with zipfile.ZipFile(DATA_ZIP) as archive:
    members = [n for n in archive.namelist() if not n.endswith("/")]
    required = []
    for fold in FOLDS:
        required.append(f"random_indices_9fold/random_indices_train_9fold_fold_{fold}.txt")
        for modality in modalities:
            prefix = "vitpose" if modality == "vitpose_kpc" else modality
            for split in ["train", "test"]:
                for kind in ["input", "output"]:
                    required.append(f"data_{modality}_9fold/{kind}_{prefix}_9fold_fold_{fold}_{split}.txt")
    for relative in required:
        matches = [n for n in members if n == relative or n.endswith("/" + relative)]
        if len(matches) != 1:
            raise FileNotFoundError(f"ZIP thiếu hoặc trùng file: {relative}")
label_modality = modalities[0]
label_prefix = "vitpose" if label_modality == "vitpose_kpc" else label_modality
labels = loadtxt(f"../data/data_{label_modality}_9fold/output_{label_prefix}_9fold_fold_{FOLDS[0]}_train.txt")
print("Đã kiểm tra đủ tên file cho các fold đã chọn.")
print("Số mẫu train của fold đầu:", labels.size)
print("Nhãn:", sorted(set(labels.reshape(-1).tolist())))


## Huấn luyện

Mỗi fold có thư mục kết quả và `train.log` riêng. Code giữ batch size 4, cách chia validation, chuẩn hóa và kiến trúc của tác giả.

Fold chạy xong được đánh dấu để bỏ qua khi chạy lại cùng cấu hình. **Fold bị ngắt sẽ chạy lại từ đầu**: chưa hỗ trợ resume optimizer/epoch. Checkpoint tốt nhất được ghi sau mỗi run hoàn tất.


In [ ]:
for fold in FOLDS:
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(exist_ok=True)
    marker = fold_dir / "completed.json"
    if SKIP_COMPLETED_FOLDS and marker.exists() and list(fold_dir.glob("*.pt")):
        print(f"Bỏ qua fold {fold}: đã hoàn tất.")
        continue
    # Xóa dấu hoàn tất cũ trước khi chủ động chạy lại fold.
    marker.unlink(missing_ok=True)
    env = os.environ.copy()
    env.update(FIDGETY_DATA_ZIP=DATA_ZIP, FIDGETY_FOLDS=str(fold),
               FIDGETY_RUNS=str(RUNS), FIDGETY_EPOCHS=str(EPOCHS), PYTHONUNBUFFERED="1")
    command = [sys.executable, "-u", str(SOURCE_DIR / SCRIPTS[MODEL])]
    print(f"Bắt đầu fold {fold}; log: {fold_dir / 'train.log'}", flush=True)
    with (fold_dir / "train.log").open("a", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=fold_dir, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
    if returncode:
        raise RuntimeError(f"Fold {fold} lỗi (exit {returncode}). Xem train.log.")
    marker.write_text(json.dumps(CONFIG, indent=2))
    print(f"Đã hoàn tất fold {fold}.")


## Xem và lưu kết quả

Bảng `res_*.txt`: mỗi dòng là một run, ba cột gồm **best validation loss**, **validation accuracy**, **test accuracy**. Accuracy giữ cách tính trong code tác giả: ngưỡng 0.5 trên logits; chưa phải balanced accuracy trong bước evaluation riêng.

File `.pt` lưu model tốt nhất theo validation loss cho mỗi fold. Đừng chọn model theo test accuracy.


In [ ]:
import numpy as np
for result_file in sorted(OUTPUT_DIR.rglob("res_*.txt")):
    results = np.loadtxt(result_file, ndmin=2)
    print(result_file.relative_to(OUTPUT_DIR))
    print("  Số run:", len(results))
    print("  Mean val accuracy:", float(results[:, 1].mean()))
    print("  Mean test accuracy:", float(results[:, 2].mean()))
print("Checkpoints:")
for checkpoint in sorted(OUTPUT_DIR.rglob("*.pt")):
    print(checkpoint.relative_to(OUTPUT_DIR))


In [ ]:
import shutil
from IPython.display import FileLink, display

# Chỉ đóng gói kết quả; không chứa ZIP dữ liệu.
export_path = shutil.make_archive(str(OUTPUT_DIR.parent / EXPERIMENT_NAME), "zip", root_dir=OUTPUT_DIR)
print("Đã đóng gói:", export_path)
display(FileLink(export_path))


Sau khi chạy xong, dùng **Save Version → Save & Run All** để chạy và lưu một phiên notebook có output, hoặc tải kết quả của phiên tương tác trước khi kết thúc. Kiểm tra thư mục **Output** để chắc chắn file đã được lưu. Dataset ZIP gốc vẫn ở `/kaggle/input` và không được đóng gói cùng kết quả.

Nếu hết RAM, chạy riêng từng modality/fold và xem kích thước dữ liệu. Không tăng batch size để xử lý lỗi RAM hệ thống. Đọc ZIP giảm dung lượng ổ đĩa, không loại bỏ nhu cầu RAM.
